## Phase 1 — Environment only (start here)

**What you are doing in this phase**

1. **Install libraries** — Packages the later phases will need (Google’s Gemini client, loading secrets, images, tables).
2. **Find the project root** — The notebook lives in `Gemini/`. The rest of your data (e.g. `test_pictures/`) sits one level up. We compute that folder so paths work no matter where you opened the notebook from.
3. **Load `.env` safely** — Your API key lives in a file named `.env` at the **project root** (same folder as `.gitignore`). That file is **gitignored**, so it is not pushed to GitHub. We only check that a key is present—we do **not** call Gemini yet.
4. **Sanity-check inputs** — Confirm `test_pictures/` exists and see how many `.gif` files are there.

Run the cells under **Phase 1** in order. **Stop after Phase 1** until you are ready for Phase 2.

In [13]:
# Phase 1 — Step A: install dependencies (run once per environment; restart kernel if prompted)
%pip install -q google-generativeai python-dotenv pillow pandas openpyxl


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [17]:
# Phase 1 — Step B: project paths + load GEMINI_API_KEY from root .env (no API calls yet)
from pathlib import Path
import os

from dotenv import load_dotenv

_NB_DIR = Path.cwd().resolve()
REPO_ROOT = _NB_DIR.parent if _NB_DIR.name == "Gemini" else _NB_DIR

load_dotenv(REPO_ROOT / ".env")

API_KEY = os.getenv("GEMINI_API_KEY", "").strip()
if not API_KEY:
    raise RuntimeError(
        "GEMINI_API_KEY is missing. Put it in the file named .env (not .env.example).\n"
        f"Expected path: {REPO_ROOT / '.env'}\n"
        "Add a line: GEMINI_API_KEY=your_key\n"
        ".env is gitignored; .env.example is only a template and is NOT loaded by this notebook."
    )

_mask = f"...{API_KEY[-4:]}" if len(API_KEY) >= 4 else "(set)"
print("This notebook folder:", _NB_DIR)
print("Project root (repo):", REPO_ROOT)
print("GEMINI_API_KEY loaded:", _mask)

INPUT_DIR = REPO_ROOT / "test_pictures"
print("Input folder exists:", INPUT_DIR.is_dir(), "→", INPUT_DIR)
if INPUT_DIR.is_dir():
    _gifs = sorted(INPUT_DIR.glob("*.gif"))
    print(f"GIF files found: {len(_gifs)}")

print("\nPhase 1 complete. Do not run Phase 2 until you want the next step.")

This notebook folder: /Users/mina/Desktop/hydrology_project/Gemini
Project root (repo): /Users/mina/Desktop/hydrology_project
GEMINI_API_KEY loaded: ...MbMw
Input folder exists: True → /Users/mina/Desktop/hydrology_project/test_pictures
GIF files found: 15

Phase 1 complete. Do not run Phase 2 until you want the next step.


---
## Phase 2 — Register your key with Google’s client (uses no quota yet)

**Run Phase 1 first** (same kernel session) so `API_KEY`, `_NB_DIR`, and `INPUT_DIR` already exist.

**What this phase does**

1. **Imports** — Loads `google.generativeai` (the official Gemini SDK), plus `json`, `re`, `numpy`, `pandas`, and `PIL` for later phases. Importing alone does not call the API.

2. **`genai.configure(api_key=API_KEY)`** — Tells the library which key to use for **future** requests. This is local setup only; it does **not** send an image or burn tokens by itself.

3. **`MODEL_ID`** — Which Gemini model to use (default `gemini-2.0-flash`, or override with `GEMINI_MODEL` in `.env`).

4. **`COLUMNS_ORDER`** — Wide JSON layout from the model (day + 12 Italian months). **`MONTH_MAP`** maps those abbreviations to month numbers 1–12.

5. **`year_from_filename(stem)`** — Reads the calendar year from the GIF/XLSX stem (first 4-digit year in 1800–2100, e.g. `Q_Bari_1969_26` → 1969). Used when writing the 4-column Excel export.

6. **`OUTPUT_DIR`** — Folder `Gemini/gemini_outputs/`; Phase 4 writes `*_gemini.xlsx` there (four columns, no header).

**Billing:** Your quota is charged when the model actually **generates** a reply (Phase 3/4), not when you run this cell.

In [23]:
# Phase 2 — configure Gemini + shared constants (requires Phase 1 done)
import json
import re

import google.generativeai as genai
import numpy as np
import pandas as pd
from PIL import Image

genai.configure(api_key=API_KEY)
MODEL_ID = os.getenv("GEMINI_MODEL", "gemini-2.0-flash")

COLUMNS_ORDER = [
    "Giorno", "Gen", "Feb", "Mar", "Apr", "Mag", "Giu", "Lug", "Ago", "Set", "Ott", "Nov", "Dic"
]

MONTH_MAP = {
    "Gen": 1,
    "Feb": 2,
    "Mar": 3,
    "Apr": 4,
    "Mag": 5,
    "Giu": 6,
    "Lug": 7,
    "Ago": 8,
    "Set": 9,
    "Ott": 10,
    "Nov": 11,
    "Dic": 12,
}


def year_from_filename(stem: str) -> int:
    """Calendar year for this GIF/XLSX (one year per file). E.g. Q_Bari_1969_26 → 1969."""
    candidates = [int(x) for x in re.findall(r"\d{4}", stem)]
    for y in candidates:
        if 1800 <= y <= 2100:
            return y
    raise ValueError(f"No 4-digit year (1800–2100) found in filename: {stem!r}")


OUTPUT_DIR = _NB_DIR / "gemini_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Model:", MODEL_ID)
print("Outputs go to:", OUTPUT_DIR)

Model: gemini-2.0-flash
Outputs go to: /Users/mina/Desktop/hydrology_project/Gemini/gemini_outputs


In [10]:
for m in genai.list_models():
    if "generateContent" in m.supported_generation_methods:
        print(m.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-3-1b-it
models/gemma-3-4b-it
models/gemma-3-12b-it
models/gemma-3-27b-it
models/gemma-3n-e4b-it
models/gemma-3n-e2b-it
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3-pro-image-preview
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-tts-preview
models/gemini-robotics-er-1.5-preview
models/gemini-robotics-er-1.6-preview
models/gem

In [26]:
import google.generativeai as genai

genai.configure(api_key=API_KEY)

model = genai.GenerativeModel("gemini-2.0-flash")
response = model.generate_content("Say hello")

print(response.text)

ResourceExhausted: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash
Please retry in 16.449659564s. [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerDayPerProjectPerModel-FreeTier"
  quo

## Phase 3 — Helpers + single-table extraction

This matches **`claude_chain2/hydrologyclaudechain2.ipynb`**:

- **System prompt:** same *elite data auditor* role, **semantic identification** of the **day column** (header may be Giorno, Giorni, or other; not assumed), **two-pass audit**, **strict JSON** with canonical key `Giorno` for output.
- **User line:** same wording as chain2 (*extract daily discharge … correspond to GIORNO* / raw JSON only).
- **Extras for accuracy (toward ~99%):** bracket/comma/empty-cell rules, digit confusion notes; JSON uses **`null` for empty cells** and **number `0` for true zero** so Excel shows blanks vs zeros correctly.
- **Post-processing:** same idea as chain2—**enforce `COLUMNS_ORDER`**, then normalize cells; JSON extraction tries a **tight** `[{...},...]` match first, then the same **greedy** `\[.*\]` fallback chain2 uses.

`extract_table_gemini` is where **Gemini quota** is used (one call per GIF per pass). **Phase 4** can run a **multi-pass** flow: extract → compare to manual XLSX → feedback prompt → optional second extraction.


In [24]:
# Phase 3 — functions
# Aligned with claude_chain2/hydrologyclaudechain2.ipynb: same auditor role, semantic ID, two-pass audit,
# exact JSON key order + "raw JSON only". Extra rules added for 99% target: brackets, commas, 0 vs null, digit hints.
# Post-processing: chain2 uses greedy \[.*\] — here we prefer a tight JSON-array match, then fall back.

import calendar

from pathlib import Path

SYSTEM_INSTRUCTION = """You are an elite Data Auditor specialized in historical hydrological yearbooks.
Your mission is to transcribe the daily discharge table with 100% precision.

### 1. SEMANTIC IDENTIFICATION
- IDENTIFY THE DAY COLUMN (do not assume the header text): On the scan it may be labeled **Giorno**, **Giorni**, **G.**, **Day**, **D.**, other Italian/English abbreviations, or **no header**—use layout and the fact that **one column lists day-of-month numbers 1..31** (one row per day) aligned horizontally with the month columns.
- IDENTIFY MONTH COLUMNS: 12 columns (Gennaio to Dicembre or abbreviations Gen..Dic).
- TABLE BOUNDARIES: Focus ONLY on the 31×12 grid. Ignore summary rows (Media, Massima, Minima, Totale, Totali).

### 2. MANDATORY TWO-PASS AUDIT
- PASS 1 — VERTICAL INTEGRITY: Count numeric entries per month in the 31-day block; counts must match month length (28/29/30/31).
- PASS 2 — HORIZONTAL ALIGNMENT: The value for day N must sit on the same row as the number N in the day column (e.g. day 15 aligns across all months).

### 3. READING RULES (accuracy-critical)
- Brackets like [12.5] or [0.65] are punctuation, NOT digits. Never read '[' as '1', '3', or '6'. Do not turn [0.65] into 6.65—extract only the numeric literal inside.
- Commas may be decimal separators (12,34 → 12.34).
- Blank, '.', or '-' cells with no numeric discharge: use JSON **null** (empty cell in output). Use JSON number **0** only when the scan shows a true zero discharge.

### 4. DIGITS (common scan errors)
- 2 vs 9: '2' often has a flat horizontal base; '9' does not match that base shape.
- 3 vs 8: check loop closure in blurry regions.

### 5. OUTPUT (STRICT)
- Return ONLY a JSON array of exactly 31 objects. No markdown, no prose before or after.
- **Valid JSON:** The output MUST be a JSON array that `json.loads` can parse without errors. Do **not** use trailing commas after the last property or array element. Do **not** include `//` or `/* */` comments or any non-JSON text.
- Each object MUST use these EXACT keys in order: "Giorno","Gen","Feb","Mar","Apr","Mag","Giu","Lug","Ago","Set","Ott","Nov","Dic".
- Always use the key **"Giorno"** for the day-of-month column in JSON (even if the printed table header says Giorni or something else). "Giorno" must be integers 1..31, one row per calendar day.
- **Exactly 31 rows:** Return all days 1..31; do not omit a row even if every month value for that day is empty (use **null** for those cells).
- **No guessing:** If a cell is unclear or unreadable, do not invent a number—use JSON **null**.
- **No inference from neighbors:** Do not infer missing values based on neighboring days or patterns (e.g. do not fill gaps by extrapolation or repeating nearby numbers)—use **null** unless the scan clearly shows a digit.
- **No row shifting:** Never move a value to a different day row. Each discharge value must strictly belong to the row for that calendar day. If horizontal alignment is uncertain, prioritize correctness of which value pairs with which **Giorno** number.
- **Do not change** transcribed numbers that already match the scan unless applying the rules above forces a correction (e.g. illegal JSON, wrong row, or blank vs zero).
- Every month key must appear in every object. Each month value must be a JSON number (including 0 for real zero) or JSON **null** for empty/missing—never use 0 for a visually blank cell."""

# Same core user instruction as claude_chain2/hydrologyclaudechain2.ipynb
USER_INSTRUCTION_DEFAULT = (
    "Extract the daily discharge data from this image for all months and all 31 days. "
    "Identify which column is the day column (values 1..31) regardless of its printed label (Giorno, Giorni, etc.). "
    "Output must be valid JSON (parseable by json.loads): no trailing commas, no comments, no extra text. "
    "Do not guess unreadable cells (use null). Do not infer missing values from neighboring days or patterns. "
    "Never shift values between rows. Return ONLY the raw JSON array with keys Giorno, Gen, Feb, ... Dic as specified."
)


def normalize_table_cell_optional(v):
    """null/blank → NaN (empty Excel cell); real 0 → 0.0."""
    if v is None:
        return np.nan
    if isinstance(v, (int, np.integer)):
        return float(v)
    if isinstance(v, (float, np.floating)):
        return np.nan if pd.isna(v) else float(v)
    s = str(v).strip()
    if s == "" or s.lower() in {"null", "none", "nan", "-", "—", "–", ".", ".."}:
        return np.nan
    s = s.replace(",", ".")
    s = re.sub(r"^\[+|\]+$", "", s)
    s = s.strip("()[] ")
    s = s.replace("*", "").strip()
    if s == "" or s == ".":
        return np.nan
    m = re.search(r"-?\d+(?:\.\d+)?", s)
    if not m:
        return np.nan
    try:
        return float(m.group(0))
    except ValueError:
        return np.nan


def extract_json_array(raw_text: str):
    """Prefer tight JSON array; fall back to chain2-style greedy bracket scan."""
    m = re.search(r"\[\s*\{.*\}\s*\]", raw_text, re.DOTALL)
    if m:
        return m.group(0)
    m = re.search(r"\[.*\]", raw_text, re.DOTALL)
    return m.group(0) if m else None


def _column_values_are_days_1_to_31(series: pd.Series) -> bool:
    g = pd.to_numeric(series, errors="coerce")
    if g.isna().any():
        return False
    try:
        days = sorted(int(x) for x in g.tolist())
    except (ValueError, TypeError):
        return False
    return days == list(range(1, 32))


def resolve_day_column_name(df: pd.DataFrame):
    """Scan header may say Giorni, G., Day, etc.; pick the day column by name or by values 1..31."""
    month_keys = set(COLUMNS_ORDER[1:])
    for c in df.columns:
        if str(c).strip().lower() == "giorno":
            return c
    if "Giorno" in df.columns:
        return "Giorno"
    day_aliases = frozenset({"giorni", "g.", "day", "days", "d.", "dia", "jour", "nr", "num", "numero", "n.", "n°", "nº"})
    for c in df.columns:
        if c in month_keys:
            continue
        raw = str(c).strip().lower()
        key = raw.replace("°", "").replace("º", "")
        if raw in day_aliases or key in day_aliases or key.startswith("numero") or "giorno" in raw:
            return c
    for c in df.columns:
        if c in month_keys:
            continue
        if _column_values_are_days_1_to_31(df[c]):
            return c
    return None


def validate_parsed_wide_df(df: pd.DataFrame):
    if df is None or df.empty:
        return False, "empty"
    if len(df) != 31:
        return False, f"rows={len(df)}"
    if "Giorno" not in df.columns:
        return False, "no Giorno"
    g = pd.to_numeric(df["Giorno"], errors="coerce")
    if g.isna().any():
        return False, "Giorno NaN"
    days = sorted(int(x) for x in g.tolist())
    if days != list(range(1, 32)):
        return False, f"Giorno {days[:10]}"
    return True, "ok"


def wide_df_from_json(json_data: list) -> pd.DataFrame:
    """Normalize day column to 'Giorno', then enforce month columns + order."""
    df = pd.DataFrame(json_data)
    day_col = resolve_day_column_name(df)
    if day_col is not None and day_col != "Giorno":
        df = df.rename(columns={day_col: "Giorno"})
    for col in COLUMNS_ORDER:
        if col not in df.columns:
            df[col] = np.nan
    df = df[COLUMNS_ORDER].copy()
    df["Giorno"] = pd.to_numeric(df["Giorno"], errors="coerce").astype(int)
    for col in COLUMNS_ORDER[1:]:
        df[col] = [normalize_table_cell_optional(x) for x in df[col].tolist()]
    return df.sort_values("Giorno").reset_index(drop=True)


def wide_to_long_export_df(df_wide: pd.DataFrame, year: int) -> pd.DataFrame:
    """Chronological long table: year, month, day, value — valid days only (no Feb 31, etc.)."""
    df_idx = df_wide.set_index("Giorno")
    rows = []
    for month_name in COLUMNS_ORDER[1:]:
        m = MONTH_MAP[month_name]
        max_d = calendar.monthrange(year, m)[1]
        for d in range(1, max_d + 1):
            val = df_idx.loc[d, month_name]
            rows.append([year, m, d, val])
    return pd.DataFrame(rows)


def load_image_for_gemini(path: Path) -> Image.Image:
    im = Image.open(path)
    if getattr(im, "n_frames", 1) > 1:
        im.seek(0)
    return im.convert("RGB")


def extract_table_gemini(image_path: Path, extra_user_text: str = "") -> str:
    img = load_image_for_gemini(image_path)
    model = genai.GenerativeModel(MODEL_ID, system_instruction=SYSTEM_INSTRUCTION)
    user_text = (extra_user_text.strip() + "\n\n") if extra_user_text else ""
    user_text += USER_INSTRUCTION_DEFAULT
    response = model.generate_content(
        [user_text, img],
        generation_config={"temperature": 0, "max_output_tokens": 16384},
    )
    text = getattr(response, "text", None) or ""
    if not text and response.candidates:
        parts = response.candidates[0].content.parts
        text = "".join(getattr(p, "text", "") for p in parts)
    return text


def parse_and_validate_response(raw: str):
    js = extract_json_array(raw)
    if not js:
        return None, "no_json"
    df = wide_df_from_json(json.loads(js))
    ok, reason = validate_parsed_wide_df(df)
    if not ok:
        return None, reason
    return df, "ok"


# --- Multi-pass: compare to manual GT + feedback prompt for pass 3 (optional Phase 4) ---
MP_TREAT_MANUAL_ZERO_AS_EMPTY = True  # must match Phase 5 TREAT_MANUAL_ZERO_AS_EMPTY


def _mp_manual_empty(v, treat_zero: bool) -> bool:
    if v is None:
        return True
    if isinstance(v, (float, np.floating)) and pd.isna(v):
        return True
    s = str(v).strip()
    if s == "" or s.lower() in ("nan", "none", "nat", "#n/a", "-"):
        return True
    try:
        x = float(str(v).replace(",", "."))
        if treat_zero and abs(x) < 1e-12:
            return True
    except (ValueError, TypeError):
        return True
    return False


def _mp_model_empty(v) -> bool:
    if v is None:
        return True
    if isinstance(v, (float, np.floating)) and pd.isna(v):
        return True
    return False


def _mp_values_match(manual_val, model_val, atol: float, treat_zero: bool) -> bool:
    """When treat_zero: manual 0/blank vs model 0 or empty counts as match."""
    me = _mp_manual_empty(manual_val, treat_zero)
    mde = _mp_model_empty(model_val)
    if me and mde:
        return True
    if treat_zero and me:
        try:
            fv = float(str(model_val).replace(",", "."))
            if abs(fv) < 1e-12:
                return True
        except (TypeError, ValueError):
            pass
    if me ^ mde:
        return False
    try:
        return np.isclose(float(manual_val), float(model_val), atol=atol)
    except (ValueError, TypeError):
        return False


def _mp_load_manual_long(manual_path: Path, year: int) -> pd.DataFrame:
    raw = pd.read_excel(manual_path, header=None)
    raw = raw.iloc[:, :4].copy()
    raw.columns = ["Y", "M", "D", "V"]
    y = pd.to_numeric(raw["Y"], errors="coerce")
    mask = np.isclose(y, float(year), rtol=0, atol=0.001) | (y == year)
    sub = raw.loc[mask, ["M", "D", "V"]].copy()
    sub["M"] = pd.to_numeric(sub["M"], errors="coerce")
    sub["D"] = pd.to_numeric(sub["D"], errors="coerce")
    sub = sub.dropna(subset=["M", "D"])
    sub["M"] = sub["M"].astype(int)
    sub["D"] = sub["D"].astype(int)
    return sub


def _mp_valid_month_days(year: int):
    out = []
    for m in range(1, 13):
        for d in range(1, calendar.monthrange(year, m)[1] + 1):
            out.append((m, d))
    return out


def mp_compare_wide_to_manual(
    df_wide: pd.DataFrame,
    base_name: str,
    year: int,
    atol: float = 1e-3,
) -> dict | None:
    """Compare model wide table to `test_pictures/{base}.xlsx`. None if manual missing."""
    man_path = REPO_ROOT / "test_pictures" / f"{base_name}.xlsx"
    if not man_path.is_file():
        return None
    long_m = wide_to_long_export_df(df_wide, year)
    manual = _mp_load_manual_long(man_path, year)
    man_map = {}
    for _, r in manual.iterrows():
        man_map[(int(r["M"]), int(r["D"]))] = r["V"]
    mod_map = {}
    for _, r in long_m.iterrows():
        mod_map[(int(r["M"]), int(r["D"]))] = r["V"]
    pairs = _mp_valid_month_days(year)
    ok = 0
    bad = []
    tz = MP_TREAT_MANUAL_ZERO_AS_EMPTY
    for m, d in pairs:
        mv = man_map.get((m, d), None)
        modv = mod_map.get((m, d), np.nan)
        if _mp_values_match(mv, modv, atol, tz):
            ok += 1
        else:
            bad.append((m, d, mv, modv))
    n = len(pairs)
    return {
        "matches": ok,
        "errors": n - ok,
        "accuracy_pct": 100.0 * ok / n if n else 0.0,
        "mismatches": bad,
        "man_map": man_map,
        "mod_map": mod_map,
        "days_compared": n,
    }


def mp_classify_errors(cmp: dict, year: int, atol: float = 1e-3) -> set[str]:
    """High-level category tags only (used for Pass 3 scope)."""
    bad = cmp["mismatches"]
    man_map = cmp["man_map"]
    tz = MP_TREAT_MANUAL_ZERO_AS_EMPTY
    cats: set[str] = set()
    if not bad:
        return cats
    nb = len(bad)

    shift_fwd = shift_bwd = 0
    for m, d, mv, modv in bad:
        if _mp_model_empty(modv):
            continue
        try:
            fv = float(modv)
        except (TypeError, ValueError):
            continue
        md = calendar.monthrange(year, m)[1]
        if d + 1 <= md:
            ref = man_map.get((m, d + 1))
            if not _mp_manual_empty(ref, tz) and _mp_values_match(ref, fv, max(atol, 0.05), tz):
                shift_fwd += 1
        if d > 1:
            ref = man_map.get((m, d - 1))
            if not _mp_manual_empty(ref, tz) and _mp_values_match(ref, fv, max(atol, 0.05), tz):
                shift_bwd += 1
    if shift_fwd / nb >= 0.15 or shift_bwd / nb >= 0.15:
        cats.add("day_misalignment")

    bracket_like = 0
    for m, d, mv, modv in bad:
        if _mp_manual_empty(mv, tz) or _mp_model_empty(modv):
            continue
        try:
            a, b = float(mv), float(modv)
        except (TypeError, ValueError):
            continue
        if a <= 0:
            continue
        ratio = b / a if abs(a) > 1e-12 else 0.0
        if (0.05 < a < 4.0) and (5.0 < ratio < 25.0 or abs(b - 10.0 * a) < max(0.15, 0.4 * abs(a))):
            bracket_like += 1
    if bracket_like / nb >= 0.12:
        cats.add("bracket_misreading")

    empty_mm = 0
    for m, d, mv, modv in bad:
        if _mp_manual_empty(mv, tz) ^ _mp_model_empty(modv):
            empty_mm += 1
    if empty_mm / nb >= 0.12:
        cats.add("empty_vs_zero")

    if not cats:
        cats.add("numeric_or_other")

    return cats


def mp_build_feedback_prompt(categories: set, old_error_count: int) -> str:
    """Short Pass-3 user text: categories to fix only + global rules + strict JSON."""
    order = ("day_misalignment", "bracket_misreading", "empty_vs_zero", "numeric_or_other")
    short = {
        "day_misalignment": "row/day alignment",
        "bracket_misreading": "brackets & decimals",
        "empty_vs_zero": "null vs 0",
        "numeric_or_other": "other reads",
    }
    fix_lines = [f"- {short[k]}" for k in order if k in categories]
    body = [
        "PASS 3 — targeted correction (prior output vs reference).",
        f"Mismatching days (approx.): {old_error_count}.",
        "Fix ONLY these issue types (copy unchanged cells verbatim; do not regenerate the whole table):",
        *(fix_lines or ["- (review flagged mismatches only)"]),
        "",
        "Do not change values that are already correct.",
        "Always respect:",
        "- row/day alignment if fixing alignment issues",
        "- brackets/punctuation (not digits) if fixing bracket issues",
        "- JSON null = empty cell; 0 = true zero only",
        "",
        "Return ONLY a valid JSON array. No explanations, no markdown.",
        "31 objects; keys in order: Giorno, Gen, Feb, Mar, Apr, Mag, Giu, Lug, Ago, Set, Ott, Nov, Dic.",
    ]
    return "\n".join(body)


## Phase 4 — Batch over all GIFs

Loops `test_pictures/*.gif`, saves `Gemini/gemini_outputs/*_gemini.xlsx` as **four columns, no header**: year (from filename) | month | day | value — only **valid calendar days** for that year.

**Multi-pass (optional):** If `ENABLE_MULTIPASS_GT` is True and `test_pictures/{same_name}.xlsx` exists, pass 2 compares to ground truth. **Pass 3 runs only when `error_count > 5`** (cost control). Pass 3 output is accepted **only if** `new_error_count < old_error_count` (error-based selection). `MP_TREAT_MANUAL_ZERO_AS_EMPTY` should match Phase 5.

In [25]:
# Phase 4 — batch (requires Phases 1–3)
if not INPUT_DIR.is_dir():
    raise FileNotFoundError(f"Missing {INPUT_DIR}")

# Pass 2–3: compare to test_pictures/{base}.xlsx when present and optionally re-extract with feedback
ENABLE_MULTIPASS_GT = True
MP_ATOL = 1e-3
MP_PASS3_ERROR_THRESHOLD = 5  # run Pass 3 only when errors > this (i.e. >5 mismatches)
MANUAL_GT_DIR = REPO_ROOT / "test_pictures"

gifs = sorted(INPUT_DIR.glob("*.gif"))
print(f"Found {len(gifs)} GIF(s)")

for gif in gifs:
    base = gif.stem
    out_xlsx = OUTPUT_DIR / f"{base}_gemini.xlsx"
    print(f"Processing {gif.name}...")
    try:
        raw = extract_table_gemini(gif)
        df, status = parse_and_validate_response(raw)
        if df is None:
            print(f"  retry ({status})...")
            raw = extract_table_gemini(
                gif,
                extra_user_text=f"Invalid prior output: {status}. Return exactly 31 rows, Giorno 1..31.",
            )
            df, status = parse_and_validate_response(raw)
        if df is None:
            print(f"  SKIP after retry: {status}")
            continue
        year = year_from_filename(base)

        comp1 = None
        if ENABLE_MULTIPASS_GT and (MANUAL_GT_DIR / f"{base}.xlsx").is_file():
            comp1 = mp_compare_wide_to_manual(df, base, year, atol=MP_ATOL)
            if comp1 is not None:
                e1 = comp1["errors"]
                print(f"  pass2 vs GT: {comp1['accuracy_pct']:.2f}% ({e1} err)")
                if e1 > MP_PASS3_ERROR_THRESHOLD:
                    cats = mp_classify_errors(comp1, year, atol=MP_ATOL)
                    print(f"  pass3 categories: {cats}")
                    fb = mp_build_feedback_prompt(cats, e1)
                    raw3 = extract_table_gemini(gif, extra_user_text=fb)
                    df3, st3 = parse_and_validate_response(raw3)
                    if df3 is not None:
                        comp3 = mp_compare_wide_to_manual(df3, base, year, atol=MP_ATOL)
                        e3 = comp3["errors"] if comp3 is not None else None
                        if comp3 is not None and e3 < e1:
                            df = df3
                            print(f"  pass3 accepted: errors {e1} -> {e3}")
                        else:
                            print(f"  pass3 rejected (errors {e3} not < {e1}), keeping pass1")
                    else:
                        print(f"  pass3 parse failed ({st3}), keeping pass1")
                elif e1 > 0:
                    print("  pass3 skipped (errors <= 5)")

        df_out = wide_to_long_export_df(df, year)
        df_out.to_excel(out_xlsx, index=False, header=False)
        print(f"  saved {out_xlsx.name} (year={year}, {len(df_out)} rows, no header)")
    except Exception as e:
        print(f"  ERROR: {e}")

print("Done.")

Found 15 GIF(s)
Processing Q_Bari_1969_26.gif...
  ERROR: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash
Please retry in 40.39504169s. [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_input_token_count"
  quota_id: "GenerateC

## Phase 5 — Compare to manual ground truth (empty vs zero)

Each **Gemini output** (`*_gemini.xlsx`) has **four columns, no header**: **year** (same on every row, from the filename) | **month** | **day** | **value**. An empty cell means missing data; **0** means a true numeric zero.

In **manual** spreadsheets, “empty” may appear as `NaN`, strings like `nan` / `Nan` / `NAN`, or a **missing row**; these are treated as matching an empty model cell.

- **Manual:** “empty” = invalid, nan strings, blank cells, or absent row. If `TREAT_MANUAL_ZERO_AS_EMPTY` is **True** (default), numeric **0** in the manual is also treated as empty; if **False**, manual **0** is a real zero.
- **Model:** “empty” = `NaN` only; **0** is a real zero discharge.

Comparison uses only **valid calendar days** for that year (365 or 366 days). Extra rows in the manual file (e.g. multiple years on one sheet) are filtered by the **year parsed from the filename**.

**Prerequisite:** Run Phase 4 so `gemini_outputs/*_gemini.xlsx` exists. Keep `TREAT_MANUAL_ZERO_AS_EMPTY` and `MP_TREAT_MANUAL_ZERO_AS_EMPTY` **identical** (default: `True`).

**Output:** `gemini_vs_manual_accuracy.xlsx` with sheet **summary** (accuracy and error count per file) and sheet **mismatches** (per error: file name, month, day, manual value, Gemini value).



In [ ]:
# Phase 5 — Compare Gemini long XLSX (4 cols, no header) to manual ground truth
import calendar

import numpy as np
import pandas as pd

# If True: in MANUAL files only, numeric 0 is treated as "empty" (for legacy sheets where 0 meant blank).
# Default False: manual 0 compares as real zero, same as model 0.
TREAT_MANUAL_ZERO_AS_EMPTY = True  # must match MP_TREAT_MANUAL_ZERO_AS_EMPTY in Phase 3

MANUAL_DIR = REPO_ROOT / "test_pictures"
GEMINI_OUT_DIR = OUTPUT_DIR  # Phase 4: year|month|day|value, no header


def _manual_cell_is_empty(v) -> bool:
    if v is None:
        return True
    if isinstance(v, (float, np.floating)) and pd.isna(v):
        return True
    s = str(v).strip()
    if s == "" or s.lower() in ("nan", "none", "nat", "#n/a", "-"):
        return True
    try:
        x = float(str(v).replace(",", "."))
        if TREAT_MANUAL_ZERO_AS_EMPTY and abs(x) < 1e-12:
            return True
    except (ValueError, TypeError):
        return True
    return False


def _model_cell_is_empty(v) -> bool:
    """Gemini wide: empty = NaN only; 0 is a real zero (not empty)."""
    if v is None:
        return True
    if isinstance(v, (float, np.floating)) and pd.isna(v):
        return True
    return False


def values_match_ground_truth(manual_val, model_val, atol: float = 1e-3) -> bool:
    """Same logic as Phase 3 _mp_values_match (incl. treat-zero vs model 0)."""
    tz = TREAT_MANUAL_ZERO_AS_EMPTY
    me = _manual_cell_is_empty(manual_val)
    mde = _model_cell_is_empty(model_val)
    if me and mde:
        return True
    if tz and me:
        try:
            fv = float(str(model_val).replace(",", "."))
            if abs(fv) < 1e-12:
                return True
        except (TypeError, ValueError):
            pass
    if me ^ mde:
        return False
    try:
        return np.isclose(float(manual_val), float(model_val), atol=atol)
    except (ValueError, TypeError):
        return False


def load_four_col_long_for_year(path: Path, year: int) -> pd.DataFrame:
    """First 4 columns: year, month, day, value — same layout for manual and Gemini exports."""
    raw = pd.read_excel(path, header=None)
    raw = raw.iloc[:, :4].copy()
    raw.columns = ["Y", "M", "D", "V"]
    y = pd.to_numeric(raw["Y"], errors="coerce")
    mask = np.isclose(y, float(year), rtol=0, atol=0.001) | (y == year)
    sub = raw.loc[mask, ["M", "D", "V"]].copy()
    sub["M"] = pd.to_numeric(sub["M"], errors="coerce")
    sub["D"] = pd.to_numeric(sub["D"], errors="coerce")
    sub = sub.dropna(subset=["M", "D"])
    sub["M"] = sub["M"].astype(int)
    sub["D"] = sub["D"].astype(int)
    return sub


def valid_month_days(year: int):
    out = []
    for m in range(1, 13):
        for d in range(1, calendar.monthrange(year, m)[1] + 1):
            out.append((m, d))
    return out


def accuracy_one_table(base_name: str, atol: float = 1e-3):
    year = year_from_filename(base_name)

    wide_path = GEMINI_OUT_DIR / f"{base_name}_gemini.xlsx"
    man_path = MANUAL_DIR / f"{base_name}.xlsx"
    if not wide_path.is_file():
        raise FileNotFoundError(wide_path)
    if not man_path.is_file():
        raise FileNotFoundError(man_path)

    long_m = load_four_col_long_for_year(wide_path, year)
    manual = load_four_col_long_for_year(man_path, year)

    man_map = {}
    for _, r in manual.iterrows():
        man_map[(int(r["M"]), int(r["D"]))] = r["V"]

    mod_map = {}
    for _, r in long_m.iterrows():
        mod_map[(int(r["M"]), int(r["D"]))] = r["V"]

    pairs = valid_month_days(year)
    ok = 0
    bad = []
    for m, d in pairs:
        mv = man_map.get((m, d), None)  # missing row => None => empty
        modv = mod_map.get((m, d), np.nan)
        if values_match_ground_truth(mv, modv, atol=atol):
            ok += 1
        else:
            bad.append((m, d, mv, modv))

    n = len(pairs)
    acc = 100.0 * ok / n if n else 0.0
    return {
        "table": base_name,
        "year": year,
        "days_compared": n,
        "matches": ok,
        "errors": n - ok,
        "accuracy_pct": acc,
        "mismatches": bad,
    }


def _fmt_val(v):
    if v is None:
        return ""
    if isinstance(v, (float, np.floating)) and pd.isna(v):
        return ""
    return v


# --- Run all benchmarks that have both files ---
results = []
for p in sorted(GEMINI_OUT_DIR.glob("*_gemini.xlsx")):
    base = p.name.replace("_gemini.xlsx", "")
    man = MANUAL_DIR / f"{base}.xlsx"
    if not man.is_file():
        print("skip (no manual):", base)
        continue
    try:
        r = accuracy_one_table(base)
        results.append(r)
        print(
            f"{base}: accuracy {r['accuracy_pct']:.2f}%  |  errors {r['errors']} / {r['days_compared']} days"
        )
        if r["errors"]:
            locs = [f"M{mm:02d}/{dd:02d}" for mm, dd, _, _ in r["mismatches"]]
            preview = ", ".join(locs[:24])
            if len(locs) > 24:
                preview += f" … (+{len(locs) - 24} more)"
            print(f"  mismatch days: {preview}")
    except Exception as e:
        print(f"{base}: ERROR {e}")

if results:
    rep = pd.DataFrame([{k: v for k, v in x.items() if k != "mismatches"} for x in results])
    rows = []
    for x in results:
        b, y = x["table"], x["year"]
        for mm, dd, mv, gv in x["mismatches"]:
            rows.append(
                {
                    "file": b,
                    "year": y,
                    "month": mm,
                    "day": dd,
                    "manual": _fmt_val(mv),
                    "gemini": _fmt_val(gv),
                }
            )
    df_bad = pd.DataFrame(rows)
    if df_bad.empty:
        df_bad = pd.DataFrame(
            columns=["file", "year", "month", "day", "manual", "gemini"]
        )
    out_xlsx = _NB_DIR / "gemini_vs_manual_accuracy.xlsx"
    with pd.ExcelWriter(out_xlsx, engine="openpyxl") as w:
        rep.to_excel(w, sheet_name="summary", index=False)
        df_bad.to_excel(w, sheet_name="mismatches", index=False)
    print("\\nSaved:", out_xlsx, "(sheets: summary, mismatches)")
else:
    print("No results — run Phase 4 first.")

